In [0]:
from pyspark.sql.functions import current_timestamp, col, split, element_at, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
CATALOG_NAME = 'beverage_sales'
SCHEMA_NAME = 'bronze'
TABLE_NAME = 'channel_group'

FILE_PATH = '/Volumes/beverage_sales/bronze/raw_data/abi_bus_case1_beverage_channel_group_20210726.csv'

In [0]:
SCHEMA = StructType(
    [
        StructField('trade_chnl_desc', StringType(), True),
        StructField('trade_group_desc', StringType(), True),
        StructField('trade_type_desc', StringType(), True)
    ]
)

In [0]:
df = spark\
    .read\
    .format('csv')\
    .schema(SCHEMA)\
    .option('header', True)\
    .option('encoding', 'UTF-8')\
    .load(FILE_PATH)\
    .withColumn('source_dir', regexp_replace(col('_metadata.file_path'), '/[^/]+$', ''))\
    .withColumn('source_file', element_at(split(col('_metadata.file_path'), '/'), -1))\
    .withColumn('ingestion_timestamp', current_timestamp())

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}')